# UTEC CS5363 - Planificación y Toma de Decisiones en IA (2026-2)
## Proyecto 2: Pipeline de Generación con ControlNet Canny y Stable Diffusion (GPU / Google Colab)

**Grupo 7:**
- José Guillermo Gálvez Pacori
- Matías Alonso Meneses Roncal
- Alejandro Joel Ore García

Este notebook permite ejecutar el **Nivel Intermedio** en una GPU con CUDA (como una GPU T4 gratuita en Google Colab) para sintetizar las variaciones fotorrealistas con cambio completo de vestimenta y fondo utilizando el condicionamiento estructural de ControlNet.

### 1. Instalación de Dependencias con Soporte GPU

In [ ]:
!pip install -q diffusers transformers accelerate controlnet_aux safetensors opencv-python pillow

### 2. Carga de Modelos: ControlNet Canny + Stable Diffusion 1.5

In [ ]:
import torch
import cv2
import numpy as np
from PIL import Image
from diffusers import StableDiffusionControlNetPipeline, ControlNetModel, UniPCMultistepScheduler

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Dispositivo activo: {device}")

controlnet = ControlNetModel.from_pretrained(
    "lllyasviel/control_v11p_sd15_canny",
    torch_dtype=torch.float16 if device == "cuda" else torch.float32
)

pipe = StableDiffusionControlNetPipeline.from_pretrained(
    "runwayml/stable-diffusion-v1-5",
    controlnet=controlnet,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32,
    safety_checker=None
)
pipe.scheduler = UniPCMultistepScheduler.from_config(pipe.scheduler.config)
if device == "cuda":
    pipe.enable_model_cpu_offload()
print("Modelos cargados exitosamente!")

### 3. Preprocesamiento Facial y Extracción de Bordes Canny

In [ ]:
def extract_canny_edges(image_path, low_thresh=100, high_thresh=200, target_size=(512, 512)):
    img = Image.open(image_path).convert("RGB").resize(target_size)
    img_np = np.array(img)
    gray = cv2.cvtColor(img_np, cv2.COLOR_RGB2GRAY)
    blurred = cv2.GaussianBlur(gray, (3, 3), 0)
    edges = cv2.Canny(blurred, low_thresh, high_thresh)
    edges_3ch = np.stack([edges]*3, axis=-1)
    return img, Image.fromarray(edges_3ch)

# Ejemplo: suba su foto como 'retrato_base.jpg'
# original_img, canny_condition = extract_canny_edges('retrato_base.jpg')
# canny_condition.save('canny_preview.png')

### 4. Generación de Variaciones por Presets del Proyecto

In [ ]:
PRESETS = {
    "linkedin_formal": "professional portrait photo of a person wearing an elegant tailored navy blue suit, white shirt, classic tie, soft studio lighting, modern corporate office background bokeh, sharp facial features, 8k uhd",
    "linkedin_business_casual": "contemporary professional portrait, person wearing a modern navy blazer without tie over light blue open-collar shirt, soft warm window lighting, stylish coworking space blurred background",
    "whatsapp_smart_casual": "clear approachable professional profile portrait, smart polo shirt, bright daylight lighting, clean luminous blurred background, friendly genuine smile",
    "github_tech_casual": "software engineering professional profile portrait, minimalist tech hoodie, ambient studio lighting with subtle cool blue rim light accent, modern developer workstation blurred bokeh",
    "content_creator": "creative studio portrait, modern textured jacket, cinematic rim lighting, dark gradient textured studio background, hyper-detailed"
}

NEGATIVE_PROMPT = "nudity, nsfw, deformed face, asymmetrical eyes, bad anatomy, blur, bad quality, extra fingers, plastic look"

def generate_variation(canny_image, prompt, negative_prompt=NEGATIVE_PROMPT, num_steps=25, guidance=7.5):
    result = pipe(
        prompt=prompt,
        negative_prompt=negative_prompt,
        image=canny_image,
        num_inference_steps=num_steps,
        guidance_scale=guidance,
        controlnet_conditioning_scale=0.85
    ).images[0]
    return result

# Para generar el lote:
# for preset_id, prompt in PRESETS.items():
#     generated = generate_variation(canny_condition, prompt)
#     generated.save(f"output_{preset_id}.png")
#     print(f"Generado: {preset_id}")